# 05 · Graph RAG

目标：在同一份向量库上，对比两种检索组织方式：

- **Linear RAG**：`query -> vector search -> top-k chunks -> answer`
- **Graph RAG**：
  - 先做一次向量检索拿到 seed chunks
  - 从 seed chunks 抽取实体/关键词（节点）
  - 在实体共现图上做 1-hop 扩展（找“相关实体”）
  - 把扩展到的 chunks 加回候选集，再生成

> 说明：这里做的是“最小可跑、可讲清楚”的 Graph RAG，不追求工业级图谱构建（那需要更复杂的抽取、消歧、schema、增量更新）。


In [1]:
from __future__ import annotations

import os
import re
from collections import Counter, defaultdict
from dataclasses import dataclass
from pathlib import Path

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma


def resolve_project_root() -> Path:
    cwd = Path.cwd().resolve()

    for candidate in (cwd, cwd.parent):
        if (candidate / "data").exists() and (candidate / "notebooks").exists():
            return candidate

    raise FileNotFoundError("未找到项目根目录，请从 RAG_project 根目录或 notebooks 目录运行本 notebook。")



def load_project_env(project_root: Path) -> Path | None:
    for env_path in (project_root / ".env", project_root.parent / ".env"):
        if env_path.exists():
            load_dotenv(env_path, override=True)
            return env_path
    return None


PROJECT_ROOT = resolve_project_root()
ENV_FILE = load_project_env(PROJECT_ROOT)
CHROMA_DIR = PROJECT_ROOT / "data/chroma"
COLLECTION = "autel_annual_report_2024"

openai_api_key = os.getenv("OPENAI_API_KEY")
openai_base_url = os.getenv("OPENAI_BASE_URL", "https://openrouter.ai/api/v1")
embed_model = os.getenv("EMBED_MODEL", "text-embedding-3-small")
chat_model = os.getenv("CHAT_MODEL") or os.getenv("LLM_MODEL", "gpt-4o-mini")

assert CHROMA_DIR.exists(), f"找不到 Chroma 目录：{CHROMA_DIR.resolve()}（先跑 01_data_02_chunk_ingest.ipynb）"
assert openai_api_key, f"未加载 OPENAI_API_KEY（检查 {ENV_FILE or PROJECT_ROOT.parent / '.env'}）"

client_kwargs = {
    "api_key": openai_api_key,
    "base_url": openai_base_url,
}

# 当前本地 Chroma 已按 1536 维 embedding 建库；若更换 embedding 模型，请先重建 data/chroma。
emb = OpenAIEmbeddings(model=embed_model, **client_kwargs)
vs = Chroma(collection_name=COLLECTION, embedding_function=emb, persist_directory=str(CHROMA_DIR))

llm = ChatOpenAI(model=chat_model, temperature=0, **client_kwargs)
print("ready:", COLLECTION, "embed:", embed_model, "chat:", chat_model, "env:", ENV_FILE)


/opt/anaconda3/envs/voc/lib/python3.11/site-packages/requests/__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.1.0)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
/var/folders/hb/4k5shxzs4c7by7lm5s0mmgrw0000gn/T/ipykernel_83246/987073138.py:53: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  vs = Chroma(collection_name=COLLECTION, embedding_function=emb, persist_directory=str(CHROMA_DIR))


ready: autel_annual_report_2024 embed: text-embedding-3-small chat: openai/gpt-4o env: /Users/mengbai/Documents/AI-training/.env


In [2]:
# 0) baseline：Linear RAG（只做检索展示）


def retrieve(query: str, k: int = 6):
    return vs.similarity_search(query, k=k)


def show(docs, max_chars: int = 220):
    for i, d in enumerate(docs, 1):
        meta = {k: d.metadata.get(k) for k in ("chunk_id", "h1", "h2", "h3") if k in d.metadata}
        print(f"[{i}]", meta)
        print(d.page_content[:max_chars].replace("\n", " "))
        print()


Q = "道通2024年年报中，主营业务/产品线的收入结构如何？"
linear_docs = retrieve(Q, k=6)
print("=== Linear RAG: top-k ===")
show(linear_docs)


=== Linear RAG: top-k ===
[1] {'chunk_id': 797, 'h2': '(一) 收入确认', 'h3': '1. 事项描述'}
相关信息披露详见财务报表附注五(33)及七(61)。   道通科技公司的营业收入主要来自于销售汽车综合诊断产品、TPMS系列产品、ADAS系列产品、汽车电子零部件和新能源充电桩等及提供相关产品的软件云服务。2024年度，道通科技公司营业收入金额为人民币393,225.64万元，其中主营业务收入为人民币388,497.45万元，占营业收入的98.80%。

[2] {'chunk_id': 344, 'h2': '2、收入和成本分析'}
break-word;'>37.27</td><td style='text-align: center; word-wrap: break-word;'>52.98</td><td style='text-align: center; word-wrap: break-word;'>44.66</td><td style='text-align: center; word-wrap: break-word;'>增加3.62个百分点</

[3] {'chunk_id': 1874, 'h2': '6、分部信息', 'h3': '(2). 报告分部的财务信息'}
<table border=1 style='margin: auto; word-wrap: break-word;'><tr><td style='text-align: center; word-wrap: break-word;'>项目</td><td style='text-align: center; word-wrap: break-word;'>中国境内</td><td style='text-align: center

[4] {'chunk_id': 341, 'h2': '2、收入和成本分析'}
✓适用 ☐不适用   报告期内，公司实现营业收入 393,225.64 万元，同比增长 20.95%。   (1). 主营业务分行业、分产品、分地区、分销售模式情况   单位：元 币种：人民币

[5] {'chunk_id': 187, 'h3': '(一)主要会计数据'}
<table bor

In [3]:
# 1) Graph RAG
# 用“实体共现图”做 1-hop 扩展：
# - 每个 chunk 抽 N 个关键词/实体
# - chunk 内两两共现形成边
# - 先检索 seed chunks，然后从 seed 的实体出发扩展相关实体，再找更多 chunk

STOP = set("的 了 和 与 及 或 在 为 对 于 以及 本 公司 我们 其 相关 主要 通过 进行".split())


def simple_terms(text: str, top_n: int = 8) -> list[str]:
    # 极简关键词：中英文/数字串 + 2-6 长度中文片段（课堂演示够用）
    cand = []
    cand += re.findall(r"[A-Za-z][A-Za-z0-9\-_/]{2,}", text)
    cand += re.findall(r"\d{4}年|\d+\.\d+%|\d+%|\d{1,3}(?:,\d{3})*(?:\.\d+)?", text)
    cand += re.findall(r"[\u4e00-\u9fff]{2,6}", text)

    cand = [c for c in cand if c not in STOP]
    freq = Counter(cand)
    # 过滤一些太常见的噪声（只保留相对更“专有”的词）
    return [w for w, _ in freq.most_common(top_n)]


@dataclass
class GraphIndex:
    term_to_chunk_ids: dict[str, set[str]]
    term_graph: dict[str, Counter]


def build_graph_index(docs) -> GraphIndex:
    term_to_chunk_ids: dict[str, set[str]] = defaultdict(set)
    term_graph: dict[str, Counter] = defaultdict(Counter)

    for d in docs:
        cid = str(d.metadata.get("chunk_id", ""))
        terms = simple_terms(d.page_content, top_n=10)
        for t in terms:
            term_to_chunk_ids[t].add(cid)
        for i in range(len(terms)):
            for j in range(i + 1, len(terms)):
                a, b = terms[i], terms[j]
                if a == b:
                    continue
                term_graph[a][b] += 1
                term_graph[b][a] += 1

    return GraphIndex(term_to_chunk_ids=dict(term_to_chunk_ids), term_graph=dict(term_graph))


# 采样一批文档来建图（课堂版：不要全量扫库，控制时间）
SAMPLE_K = int(os.getenv("GRAPH_SAMPLE_K", "120"))
seed_for_graph = retrieve("道通 年报 业务 收入 产品", k=min(12, SAMPLE_K))
# 再随机补一些：用几个 query 拼一点覆盖（避免太偏）
extra = []
for q in ["研发 投入", "产品 线", "分部", "财务 报表", "毛利率"]:
    extra += retrieve(q, k=10)

# 去重（用 chunk_id）
uniq = {}
for d in seed_for_graph + extra:
    uniq[str(d.metadata.get("chunk_id", len(uniq)))] = d
sample_docs = list(uniq.values())[:SAMPLE_K]

G = build_graph_index(sample_docs)
print("graph terms:", len(G.term_graph), "sample_docs:", len(sample_docs))


graph terms: 191 sample_docs: 61


In [4]:
# 2) Graph 扩展检索


def graph_expand_terms(seed_docs, top_terms: int = 6, hop: int = 1, expand_per_term: int = 3) -> list[str]:
    seed_text = "\n".join(d.page_content for d in seed_docs)
    seeds = simple_terms(seed_text, top_n=top_terms)

    expanded = set(seeds)
    frontier = list(seeds)
    for _ in range(hop):
        nxt = []
        for t in frontier:
            neigh = [w for w, _ in G.term_graph.get(t, Counter()).most_common(expand_per_term)]
            for w in neigh:
                if w not in expanded:
                    expanded.add(w)
                    nxt.append(w)
        frontier = nxt

    return list(expanded)


def graph_rag_retrieve(query: str, seed_k: int = 6, final_k: int = 10):
    # step1: seed retrieval
    seed_docs = retrieve(query, k=seed_k)

    # step2: expand via graph
    terms = graph_expand_terms(seed_docs, top_terms=6, hop=1, expand_per_term=4)

    # step3: pull candidate chunk_ids by expanded terms
    candidate_ids = set()
    for t in terms:
        candidate_ids |= set(G.term_to_chunk_ids.get(t, set()))

    # step4: for demo we do a second vector search using an enriched query
    enriched_query = query + "\n" + " ".join(terms)
    docs2 = retrieve(enriched_query, k=final_k)

    return seed_docs, terms, candidate_ids, docs2


seed_docs, terms, candidate_ids, graph_docs = graph_rag_retrieve(Q, seed_k=6, final_k=10)
print("=== Graph RAG trace ===")
print("[seed] top-6")
show(seed_docs)
print("[expanded terms]", terms)
print("[candidate chunk_ids]", len(candidate_ids))
print("\n[graph-rag final top-10] (enriched query)")
show(graph_docs)


=== Graph RAG trace ===
[seed] top-6
[1] {'chunk_id': 797, 'h2': '(一) 收入确认', 'h3': '1. 事项描述'}
相关信息披露详见财务报表附注五(33)及七(61)。   道通科技公司的营业收入主要来自于销售汽车综合诊断产品、TPMS系列产品、ADAS系列产品、汽车电子零部件和新能源充电桩等及提供相关产品的软件云服务。2024年度，道通科技公司营业收入金额为人民币393,225.64万元，其中主营业务收入为人民币388,497.45万元，占营业收入的98.80%。

[2] {'chunk_id': 344, 'h2': '2、收入和成本分析'}
break-word;'>37.27</td><td style='text-align: center; word-wrap: break-word;'>52.98</td><td style='text-align: center; word-wrap: break-word;'>44.66</td><td style='text-align: center; word-wrap: break-word;'>增加3.62个百分点</

[3] {'chunk_id': 1874, 'h2': '6、分部信息', 'h3': '(2). 报告分部的财务信息'}
<table border=1 style='margin: auto; word-wrap: break-word;'><tr><td style='text-align: center; word-wrap: break-word;'>项目</td><td style='text-align: center; word-wrap: break-word;'>中国境内</td><td style='text-align: center

[4] {'chunk_id': 341, 'h2': '2、收入和成本分析'}
✓适用 ☐不适用   报告期内，公司实现营业收入 393,225.64 万元，同比增长 20.95%。   (1). 主营业务分行业、分产品、分地区、分销售模式情况   单位：元 币种：人民币

[5] {'chunk_id': 187, 'h3': '(一)主要会计数据'}

## 课堂结论怎么讲（建议）

- **Linear RAG**：像“按相似度排序的列表”，优点是简单稳定；缺点是容易漏掉“同一实体/主题的相关证据”。
- **Graph RAG**：像“从 seed 证据出发做关联扩展”，更擅长补全上下文与跨段落证据。

> 如果你想做更像论文/生产的 Graph RAG：把 `simple_terms()` 换成 LLM 抽取的实体/关系（三元组），再做消歧与 schema；并把图谱构建放到 ingestion 阶段，而不是 query 时临时建。
